# Notebook 1 — Exploration and preprocessing of the ARCADE dataset

**Goal:** understand the data, visualise the quality variability, and standardise the images through MSCN normalisation.

**Deliverable:** a list of images labelled by hand as `good` / `bad`, and an overview of the MSCN-normalised images.

## 0. Paths

In [ ]:
import os
from pathlib import Path

# Paths — dataset: $ARCADE_ROOT if set, otherwise <repo>/data/arcade
PROJECT_ROOT   = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
ARCADE_ROOT    = Path(os.environ.get("ARCADE_ROOT", PROJECT_ROOT / "data" / "arcade"))
SEG_TRAIN_PATH = ARCADE_ROOT / "dataset_phase_1/segmentation_dataset/seg_train"
SEG_VAL_PATH   = ARCADE_ROOT / "dataset_phase_1/segmentation_dataset/seg_val"

# Dossier de sortie
OUTPUT_DIR  = Path("outputs_cvae/01_Data_Exploration")
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 20   # number of images to load for the exploration

print(f"Train : {SEG_TRAIN_PATH}")
print(f"Val   : {SEG_VAL_PATH}")

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy.ndimage import uniform_filter
import random, os

random.seed(42)
np.random.seed(42)

## 2. Loading a representative sample

In [ ]:
def collect_images(root: Path, exts=(".png", ".jpg", ".jpeg")) -> list[Path]:
    """Return all image files in root/images/ or root/."""
    image_dir = root / "images" if (root / "images").exists() else root
    return sorted(p for p in image_dir.rglob("*") if p.suffix.lower() in exts)

all_train_images = collect_images(SEG_TRAIN_PATH)
all_val_images   = collect_images(SEG_VAL_PATH)

print(f"Train : {len(all_train_images)} images")
print(f"Val   : {len(all_val_images)} images")

# Random sample
sample_paths = random.sample(all_train_images, min(N_SAMPLE, len(all_train_images)))
print(f"\nSelected sample: {len(sample_paths)} images")

In [ ]:
def load_gray(path: Path) -> np.ndarray:
    """Load a greyscale image, float32 in [0, 1]."""
    img = Image.open(path).convert("L")
    return np.asarray(img, dtype=np.float32) / 255.0

sample_images = [load_gray(p) for p in sample_paths]

print(f"Image size: {sample_images[0].shape}")
print(f"Min / Max : {sample_images[0].min():.3f} / {sample_images[0].max():.3f}")

## 3. Visualising the sample — overview of the quality variability

In [ ]:
n_cols = 5
n_rows = int(np.ceil(len(sample_images) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
axes = axes.flatten()

for i, (img, path) in enumerate(zip(sample_images, sample_paths)):
    axes[i].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(path.stem, fontsize=7)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Overview of ARCADE images (seg_train)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_panorama_sample.png", dpi=100, bbox_inches="tight")
plt.show()

## 4. Basic metrics to spot degraded images visually

Three simple no-reference indicators are computed to sort the dataset:

| Indicator | What it measures | Expected on a good-quality image |
|---|---|---|
| **Laplacian variance** | Sharpness (sharp edges) | **High** value |
| **Shannon entropy** | Complexity / disorder | **Low** value (fewer artefacts) |
| **RMS contrast** | Global spread of the grey levels | **Medium** to high value |

In [ ]:
from scipy.ndimage import laplace
from skimage.measure import shannon_entropy

def laplacian_variance(img: np.ndarray) -> float:
    return float(laplace(img).var())

def rms_contrast(img: np.ndarray) -> float:
    return float(img.std())

def shannon_ent(img: np.ndarray) -> float:
    return float(shannon_entropy(img))

metrics_raw = []
for img, path in zip(sample_images, sample_paths):
    metrics_raw.append({
        "name": path.stem,
        "path": path,
        "laplacian_var": laplacian_variance(img),
        "rms_contrast":  rms_contrast(img),
        "entropy":       shannon_ent(img),
    })

# Sort by increasing sharpness: the first entries are the blurriest images
metrics_raw.sort(key=lambda x: x["laplacian_var"])

print("Blurriest images (low Laplacian variance):")
for m in metrics_raw[:5]:
    print(f"  {m['name']:30s}  lap={m['laplacian_var']:.5f}  rms={m['rms_contrast']:.4f}  H={m['entropy']:.4f}")

print("\nSharpest images (high Laplacian variance):")
for m in metrics_raw[-5:]:
    print(f"  {m['name']:30s}  lap={m['laplacian_var']:.5f}  rms={m['rms_contrast']:.4f}  H={m['entropy']:.4f}")

## 5. Side by side: good image vs bad image

This is the key figure of the report: it shows concretely the problem the project addresses.

In [ ]:
worst  = metrics_raw[0]   # blurriest / most artefacted image
best   = metrics_raw[-1]  # sharpest image

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(load_gray(worst["path"]), cmap="gray", vmin=0, vmax=1)
axes[0].set_title(
    f"Degraded image (bad)\n"
    f"Laplacian = {worst['laplacian_var']:.5f}",
    color="crimson"
)
axes[0].axis("off")

axes[1].imshow(load_gray(best["path"]), cmap="gray", vmin=0, vmax=1)
axes[1].set_title(
    f"Good-quality image (good)\n"
    f"Laplacian = {best['laplacian_var']:.5f}",
    color="forestgreen"
)
axes[1].axis("off")

fig.suptitle("Quality comparison on ARCADE", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_good_vs_bad.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Distribution of the metrics over the sample

In [ ]:
import pandas as pd

df = pd.DataFrame(metrics_raw).drop(columns=["path"])
print(df.describe().round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, color in zip(axes, ["laplacian_var", "rms_contrast", "entropy"],
                           ["steelblue", "darkorange", "mediumpurple"]):
    ax.hist(df[col], bins=10, color=color, edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("value")
    ax.set_ylabel("count")

fig.suptitle("Distribution of the raw quality metrics", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_metrics_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

## 7. MSCN normalisation (mean subtracted contrast normalised)

MSCN normalisation subtracts the local mean and divides by the local standard deviation. It solves two common problems of X-rays:
- Non-uniform illumination (the centre is brighter than the borders)
- Global contrast biases between patients

$$\hat{I}(i,j) = \frac{I(i,j) - \mu(i,j)}{\sigma(i,j) + C}$$

where $\mu$ and $\sigma$ are computed over a local window of size $w$, and $C=1$ is a stabilising constant.

In [ ]:
def mscn_normalize(img: np.ndarray, window: int = 7, C: float = 1.0) -> np.ndarray:
    """
    Apply the MSCN normalisation to a float32 image in [0,1].
    Returns an array with local mean ~0 and local variance ~1.
    """
    mu    = uniform_filter(img, size=window).astype(np.float64)
    mu_sq = uniform_filter(img ** 2, size=window).astype(np.float64)
    sigma = np.sqrt(np.maximum(mu_sq - mu ** 2, 0))
    mscn  = (img.astype(np.float64) - mu) / (sigma + C)
    return mscn.astype(np.float32)


# Test on two images
bad_raw  = load_gray(worst["path"])
good_raw = load_gray(best["path"])

bad_mscn  = mscn_normalize(bad_raw)
good_mscn = mscn_normalize(good_raw)

print(f"Image brute     — min={bad_raw.min():.3f}  max={bad_raw.max():.3f}  mean={bad_raw.mean():.3f}")
print(f"Image MSCN      — min={bad_mscn.min():.3f}  max={bad_mscn.max():.3f}  mean={bad_mscn.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

titles = [
    ("Brute — Bad",  bad_raw,   "crimson"),
    ("MSCN  — Bad",  bad_mscn,  "crimson"),
    ("Diff  — Bad",  bad_raw - np.interp(bad_raw, [bad_raw.min(), bad_raw.max()], [0, 1]),  "gray"),
    ("Brute — Good", good_raw,  "forestgreen"),
    ("MSCN  — Good", good_mscn, "forestgreen"),
    ("Diff  — Good", good_raw - np.interp(good_raw, [good_raw.min(), good_raw.max()], [0, 1]), "gray"),
]

cmaps    = ["gray", "RdBu_r", "gray", "gray", "RdBu_r", "gray"]
sym_norm = [False, True, False, False, True, False]

for ax, (title, data, color), cmap, sym in zip(axes.flatten(), titles, cmaps, sym_norm):
    if sym:
        vmax = np.abs(data).max()
        ax.imshow(data, cmap=cmap, vmin=-vmax, vmax=vmax)
    else:
        ax.imshow(data, cmap=cmap)
    ax.set_title(title, color=color, fontsize=10)
    ax.axis("off")

fig.suptitle("Effect of the MSCN normalisation", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_mscn_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Histograms before / after MSCN

Check that MSCN centres the distribution and removes the illumination biases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, img, mscn, label, color in zip(
    axes,
    [bad_raw,   good_raw],
    [bad_mscn,  good_mscn],
    ["Bad",     "Good"],
    ["crimson", "forestgreen"],
):
    ax.hist(img.ravel(),  bins=80, alpha=0.5, label="Brute",  color="gray",  density=True)
    ax.hist(mscn.ravel(), bins=80, alpha=0.7, label="MSCN",   color=color, density=True)
    ax.set_title(f"Histogramme — {label}")
    ax.set_xlabel("intensity")
    ax.set_ylabel("density")
    ax.legend()

fig.suptitle("Pixel distributions before and after MSCN normalisation", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_histograms_mscn.png", dpi=100, bbox_inches="tight")
plt.show()

## 9. Robustness of MSCN: metrics after normalisation

Check that, after MSCN, the Laplacian variance separates the images even better:

In [ ]:
metrics_mscn = []
for m in metrics_raw:
    img_raw  = load_gray(m["path"])
    img_norm = mscn_normalize(img_raw)
    metrics_mscn.append({
        "name":            m["name"],
        "laplacian_var":   laplacian_variance(img_norm),
        "rms_contrast":    rms_contrast(img_norm),
        "entropy":         shannon_ent(
                               # MSCN can be negative: remap for the entropy
                               (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min() + 1e-8)
                           ),
    })

df_mscn = pd.DataFrame(metrics_mscn)
print("Post-MSCN metrics:")
print(df_mscn.describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, color_raw, color_norm in zip(
    axes,
    ["laplacian_var", "rms_contrast", "entropy"],
    ["steelblue",    "darkorange",   "mediumpurple"],
    ["deepskyblue",  "gold",         "violet"],
):
    ax.hist(df[col],      bins=10, color=color_raw,  alpha=0.6, label="Brute",  edgecolor="white")
    ax.hist(df_mscn[col], bins=10, color=color_norm, alpha=0.7, label="MSCN",   edgecolor="white")
    ax.set_title(col)
    ax.legend(fontsize=8)

fig.suptitle("Raw vs post-MSCN metrics", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_metrics_raw_vs_mscn.png", dpi=100, bbox_inches="tight")
plt.show()

## Summary of notebook 1

| Step | Result |
|---|---|
| ARCADE loading | Greyscale float32 images [0,1] |
| Raw metrics | Laplacian variance, RMS contrast, entropy |
| Good/bad visualisation | Figure `01_good_vs_bad.png` |
| MSCN normalisation | Removes the local illumination biases |

**Next notebook:** `02_IQA_Metrics_Extraction.ipynb` — per-patch computation (4×4 grid) over the whole dataset.